# Dreamer V1 from Scratch

### Markov Decision Process (MDP)

Markov Systems are systems where the next state depends only on the current state and the action taken

A MDP models how the state of a system evolves when different actions are applied to the system.

Let:
- $S$ be the set of states in a system
- $A$ be the set of potential actions; they may modify the state of the actor, i.e. change his state to another one within $S$
- $T : (S \times A \times S) \rightarrow [0, 1]$ be a *transition function* such that $T(s, a, s') = P(s' | s, a)$ (i.e. the probability of reaching state $s'$ given that action $a$ is applied when the actor has the state $s$). Since $T$ is a probability distribution, we have that $\sum_{s' \in S} T(s, a, s') = 1$
- $r : S \times A \rightarrow \R$ be the reward function such that the actor gets reward $r(s, a)$ if it takes action $a$ at state $s$. If the reward is small, then the action is less useful for achieving the goal

Then, we define $MDP : (S, A, T, R, \gamma), \text{where } \gamma \in [0, 1)$ is the discount factor

Reward is calculated as $R(\tau) = r_0 + r_1 + r_2 + ...$ for a trajectory $\tau$, and there may be a discount factor $\gamma$ such that $R(\tau) = r_0 + \gamma r_1 + \gamma^2 r_2 + ...$

The ultimate goal is to maximize $R(\tau)$

[Bibliography](https://d2l.ai/chapter_reinforcement-learning/mdp.html)

### Partially Observable MDP

Imagine playing a video game where you can only see the image (frame). Then, since that is the only information available to you, there is no way of tracking the exact state parameters the game state registers, such as velocity and position. Thus, it breaks the Markov System presumption that given a state and an action, the next state can be reliably predicted.

In this case, we are dealing with an observation $o_t$ which holds only partial or potentially noisy information regarding the state $s_t$.

Thus, we are dealing with a POMDP when we have:
- A hidden space state $S$
- An observation space $O$ and a function which maps states to observations
- Everything else from a MDP

The approach is to create a belief state distribution $b_t = P(o_\text{1:t}, a_\text{1, t-1})$ over what the true state might be, and we update it with every new observation.

### Reinforcement Learning Agent

$a_t ~ p(a_t | o_\text{1:t}, a_\text{1, t-1})$ and $o_t, r_t ~ P(o_t, r_t | o_\text{1:t-1}, a_\text{1, t-1})$

*The goal is to develop an agent that maximizes the expected sum of the rewards* $\mathbb E_p(\sum_\text{t=1}r_t)$

### The Value Function
The expected cumulative discounted reward you'll collect starting from state $s$ and following policy $\pi$ forever.

$$V^\pi(s) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0 = s, \pi\right]$$

Basically, it says: *if I'm in state $s$ and follow policy $\pi$ forever, how much total discounted reward will I collect?* 

**Claude** says:
    *Its goal is to compress an infinite future into a single number that you can reason about. Different policies produce different value functions — a bad policy has low values everywhere, a good one has high values.*

The optimal policy $\pi^*$ is the one whose value function is maximal at every state simultaneously:

$$V^*(s)=\max⁡_\pi V^\pi(s) \;\forall s$$

### The Bellman Equation

We separate the current state from the rest of the states:
$$V^\pi(s) = r(s, \pi(s)) + \gamma \, \mathbb{E}_{s' \sim P(\cdot \mid s, \pi(s))}\left[V^\pi(s')\right]$$

Therefore, we get something like: *the value of where you are = what you get right now + discounted average value of where you'll end up*.

This is a reformulation of the value function, which reveals a recursive form that we can use for computation (can't compute an infinite sum), since $s'$ is the state immediately following $s$.

Its goal is to give you a checkable consistency condition, i.e. if your estimate of $V_\pi$ is correct, this equation must hold everywhere. If it doesn't hold somewhere, you have a concrete, computable error. This is the crucial bridge to learning.

For a finite state space, this is literally a system of linear equations (one per state). Because $0 <= \gamma < 1$, the matrix $(I−\gamma P)$ is always invertible, since $\gamma < 1$ and every eigenvalue of $P$ is smaller or equal to 1 (property of stochastic matrices; each row is a probability distribution), so all eigenvalues of $(I−\gamma P)$ are greater than 0 (so the matrix is invertible). Thus, it guarantees a unique solution: the true $V_\pi$. This uniqueness is what makes it a well-defined target.



### TD Learning (Temporal Difference Learning)

In practice, you don't know $P$ or $r$ analytically, so you can't solve the linear system directly. Instead, you sample experience and iteratively enforce the Bellman equation one step at a time.

After observing one real transition $(s,a,r,s′)$, you compute the TD target, i.e. what the Bellman equation says $V(s)$ should approximately equal to: $$\text{target} = r + \gamma \hat{V}(s′)$$

And nudge your estimate toward it:
$$\hat{V}(s) \leftarrow \hat{V}(s) + \alpha [r + \gamma \hat{V}(s′) − \hat{V}(s)]$$

We don't have access to the true $V_\pi$. The TD $\text{target} = r + \gamma \hat{V}(s′)$ is not the truth, but a bootstrap.
- $\hat(V)(s)$ is the current estimate at the state you just left
- $\gamma \hat{V}(s′)$ is a slightly better-grounded estimate, because it anchors to one real reward $r$ before using the approximation

Instead of targeting a well-known result, we're chasing a moving target that is itself an approximation. 

The term in brackets is the TD error, which calculates how much your current estimate violates the Bellman equation at this state. The Bellman operator is a contraction mapping, meaning that each application shrinks the gap between your estimate and the truth by a factor of $\gamma$*.


*) **Claude** explains:

*At every update, your estimate at $(s_2)$ is:*

$$[\hat{V}^{(n+1)}(s_2) = 1 + 0.9 \times \hat{V}^{(n)}(s_2)]$$

*The true value satisfies the same equation exactly:*

$$[V^*(s_2) = 1 + 0.9 \times V^*(s_2)]$$

*Subtract one from the other:*

$$[V^*(s_2) - \hat{V}^{(n+1)}(s_2) = 0.9 \times \left(V^*(s_2) - \hat{V}^{(n)}(s_2)\right)]$$

*The gap at step $(n+1)$ is exactly 0.9 times the gap at step $n$. Always. The reward term cancels, so the only thing that survives is $\gamma$ multiplying the old gap.*

| Component           | Goal                                                                         |
| ------------------- | ---------------------------------------------------------------------------- |
| MDP                 | Formally defines the environment: states, actions, transitions, rewards |
| Value function      | Measures how good a state is under a given policy |
| Bellman equation    | Characterizes the correct value function via a local consistency condition |
| TD learning         | Enforces that consistency from sampled data, without knowing the environment |
| Policy optimization | Uses the learned value function to improve the policy towards $\pi^*$ (optimal policy) |